# PositionManager and PositionTracker Testing

This notebook provides a comprehensive test suite for both `PositionManager` and `PositionTracker` modules within the `PositionManager` directory. It uses mocks to simulate MetaTrader 5 (MT5) interactions, allowing the tests to run without a live MT5 connection.

In [1]:
import sys
import os
import time
import json
import logging
from datetime import datetime, timezone
from unittest.mock import MagicMock, patch

# Add current directory to path so we can import the modules
sys.path.append(os.getcwd())

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
logger = logging.getLogger("TestNotebook")

## 1. Mocking MetaTrader 5

Since MT5 is a Windows-only library and requires a live terminal, we mock it entirely for these tests.

In [2]:
# Create a mock for the MetaTrader5 module
mt5 = MagicMock()

# Define necessary constants
mt5.ORDER_FILLING_FOK = 0
mt5.ORDER_TYPE_BUY = 0
mt5.ORDER_TYPE_SELL = 1
mt5.TRADE_ACTION_DEAL = 1
mt5.TRADE_ACTION_SLTP = 6
mt5.ORDER_TIME_GTC = 0
mt5.TRADE_RETCODE_DONE = 10009
mt5.TRADE_RETCODE_REQUOTE = 10004

# Inject the mock into sys.modules
sys.modules["MetaTrader5"] = mt5

print("MetaTrader5 mocked successfully.")

MetaTrader5 mocked successfully.


## 2. Testing PositionManager

The `PositionManager` is a pure executor. We'll test its `open_position`, `close_position`, and `modify_position` methods.

In [3]:
from position_manager import PositionManager

def test_position_manager_open():
    print("--- Testing PositionManager.open_position ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)
    
    # Mock symbol info tick
    mt5.symbol_info_tick.return_value = MagicMock(ask=1.1000, bid=1.0990)
    
    # Mock successful order send
    mock_result = MagicMock()
    mock_result.retcode = mt5.TRADE_RETCODE_DONE
    mock_result.order = 123456
    mock_result.price = 1.1000
    mock_result.comment = "Request executed"
    mt5.order_send.return_value = mock_result
    mt5.last_error.return_value = (1, "Success")
    
    result = pm.open_position(
        symbol="GBPUSD_i",
        direction=1,
        lot_size=0.1,
        sl_price=1.0900,
        tp_price=1.1100,
        strategy="unity"
    )
    
    print(f"Open Result: {json.dumps(result, indent=2, default=str)}")
    assert result["success"] is True
    assert result["ticket"] == 123456
    assert result["strategy"] == "unity"
    assert result["lot_size"] == 0.1
    assert result["error_code"] == 0
    print("Open position test passed!\n")

def test_position_manager_close():
    print("--- Testing PositionManager.close_position ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)
    
    # Mock position to close
    mock_pos = MagicMock()
    mock_pos.symbol = "GBPUSD_i"
    mock_pos.volume = 0.1
    mock_pos.type = mt5.ORDER_TYPE_BUY
    mock_pos.magic = 100001
    mock_pos.sl = 1.0900
    mock_pos.tp = 1.1100
    mt5.positions_get.return_value = [mock_pos]
    
    # Mock successful close
    mock_result = MagicMock()
    mock_result.retcode = mt5.TRADE_RETCODE_DONE
    mock_result.price = 1.1005
    mock_result.comment = "Closed"
    mt5.order_send.return_value = mock_result
    mt5.symbol_info_tick.return_value = MagicMock(bid=1.1005, ask=1.1006)
    
    result = pm.close_position(ticket=123456)
    
    print(f"Close Result: {json.dumps(result, indent=2, default=str)}")
    assert result["success"] is True
    assert result["lot_size"] == 0.1
    print("Close position test passed!\n")

test_position_manager_open()
test_position_manager_close()

2026-06-21 20:00:40,694 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-21 20:00:40,694 [INFO] PositionManager: OPEN SUCCESS: GBPUSD_i unity 1 0.1 SL:1.09 TP:1.11 Ticket:123456
2026-06-21 20:00:40,694 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-21 20:00:40,702 [INFO] PositionManager: CLOSE SUCCESS: Ticket 123456 Symbol GBPUSD_i Vol 0.1


--- Testing PositionManager.open_position ---
Open Result: {
  "success": true,
  "ticket": 123456,
  "symbol": "GBPUSD_i",
  "direction": 1,
  "lot_size": 0.1,
  "entry_price": 1.1,
  "sl_price": 1.09,
  "tp_price": 1.11,
  "strategy": "unity",
  "error_code": 0,
  "retcode": 10009,
  "comment": ""
}
Open position test passed!

--- Testing PositionManager.close_position ---
Close Result: {
  "success": true,
  "ticket": 123456,
  "symbol": "GBPUSD_i",
  "direction": 1,
  "lot_size": 0.1,
  "entry_price": 1.1005,
  "sl_price": 1.09,
  "tp_price": 1.11,
  "strategy": "unity",
  "error_code": 0,
  "retcode": 10009,
  "comment": "Success"
}
Close position test passed!



## 3. Testing PositionTracker

The `PositionTracker` is an observer that polls MT5 in a background thread. We'll test its ability to track positions, calculate risk/reward, and persist state.

In [4]:
from position_tracker import PositionTracker

def test_position_tracker():
    print("--- Testing PositionTracker ---")
    state_file = "test_position_state_notebook.json"
    if os.path.exists(state_file): os.remove(state_file)
    
    tracker = PositionTracker(
        magic_numbers=[100001, 100002],
        poll_interval_seconds=1,
        state_file=state_file
    )
    
    # Mock positions for tracker
    mock_pos1 = MagicMock()
    mock_pos1.ticket = 111
    mock_pos1.symbol = "GBPUSD_i"
    mock_pos1.magic = 100001
    mock_pos1.type = mt5.ORDER_TYPE_BUY
    mock_pos1.volume = 0.1
    mock_pos1.price_open = 1.1000
    mock_pos1.sl = 1.0950 # 50 pips risk
    mock_pos1.tp = 1.1100 # 100 pips reward
    mock_pos1.price_current = 1.1010
    mock_pos1.profit = 10.0
    mock_pos1.time = time.time()
    
    mt5.positions_get.return_value = [mock_pos1]
    
    # Mock symbol info for contract size
    mock_info = MagicMock()
    mock_info.trade_contract_size = 100000
    mt5.symbol_info.return_value = mock_info
    
    tracker.start()
    time.sleep(1.5) # Allow one poll cycle
    
    positions = tracker.get_open_positions()
    risk = tracker.get_open_risk()
    reward = tracker.get_open_reward()
    
    print(f"Tracked Positions: {len(positions)}")
    print(f"Total Risk: ${risk:.2f}")
    print(f"Total Reward: ${reward:.2f}")
    
    # Expected Risk: 0.0050 * 0.1 * 100000 = 50.0
    # Expected Reward: 0.0100 * 0.1 * 100000 = 100.0
    assert len(positions) == 1
    assert abs(risk - 50.0) < 0.01
    assert abs(reward - 100.0) < 0.01
    assert isinstance(positions[0]["open_time"], str)
    
    tracker.stop()
    
    # Verify persistence
    assert os.path.exists(state_file)
    with open(state_file, "r") as f:
        saved_data = json.load(f)
        assert len(saved_data["positions"]) == 1
        print("State file persisted correctly.")
    
    if os.path.exists(state_file): os.remove(state_file)
    print("PositionTracker test passed!\n")

test_position_tracker()

2026-06-21 20:00:42,074 [INFO] PositionTracker: New tracked position detected: ticket 111
2026-06-21 20:00:42,080 [INFO] PositionTracker: PositionTracker started.
2026-06-21 20:00:42,080 [INFO] PositionTracker: Poll cycle summary: 1 positions tracked, total open risk $50.00


--- Testing PositionTracker ---


2026-06-21 20:00:43,096 [INFO] PositionTracker: Poll cycle summary: 1 positions tracked, total open risk $50.00
2026-06-21 20:00:43,583 [INFO] PositionTracker: PositionTracker stopped.


Tracked Positions: 1
Total Risk: $50.00
Total Reward: $100.00
State file persisted correctly.
PositionTracker test passed!



## 4. Error Handling and Edge Cases

Testing how modules handle invalid inputs and MT5 errors.

In [5]:
def test_edge_cases():
    print("--- Testing Edge Cases ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)
    
    # 1. Invalid Direction
    res1 = pm.open_position("GBPUSD_i", 5, 0.1, 1.1, 1.2, "unity")
    assert res1["success"] is False
    assert "Invalid direction" in res1["comment"]
    
    # 2. MT5 Requote
    mt5.symbol_info_tick.return_value = MagicMock(ask=1.1000, bid=1.0990)
    mock_fail = MagicMock()
    mock_fail.retcode = mt5.TRADE_RETCODE_REQUOTE
    mock_fail.comment = "Requote"
    mt5.order_send.return_value = mock_fail
    
    res2 = pm.open_position("GBPUSD_i", 1, 0.1, 1.09, 1.11, "unity")
    assert res2["success"] is False
    assert res2["retcode"] == mt5.TRADE_RETCODE_REQUOTE
    
    print("Edge cases test passed!\n")

test_edge_cases()

2026-06-21 20:00:45,928 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-21 20:00:45,929 [ERROR] PositionManager: OPEN FAILED: GBPUSD_i unity 1 Lot:0.1 Retcode:10004 Comment:Requote


--- Testing Edge Cases ---
Edge cases test passed!



In [6]:
def test_position_manager_modify():
    print("--- Testing PositionManager.modify_position ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)

    mock_pos = MagicMock()
    mock_pos.symbol = "GBPUSD_i"
    mock_pos.magic = 100001
    mock_pos.type = mt5.ORDER_TYPE_BUY
    mock_pos.volume = 0.1
    mock_pos.price_open = 1.1000
    mock_pos.sl = 1.0900
    mock_pos.tp = 1.1100
    mt5.positions_get.return_value = [mock_pos]

    mock_result = MagicMock()
    mock_result.retcode = mt5.TRADE_RETCODE_DONE
    mock_result.comment = "Modified"
    mt5.order_send.return_value = mock_result

    # Test 1: modify SL only — TP should be preserved from MT5
    result = pm.modify_position(ticket=123456, sl_price=1.0920)
    assert result["success"] is True
    assert result["sl_price"] == 1.0920
    assert result["tp_price"] == 1.1100  # preserved from pos.tp
    
    # Verify the actual request sent to MT5 used the preserved TP
    call_args = mt5.order_send.call_args[0][0]
    assert call_args["sl"] == 1.0920
    assert call_args["tp"] == 1.1100

    # Test 2: modify TP only — SL preserved
    result = pm.modify_position(ticket=123456, tp_price=1.1200)
    call_args = mt5.order_send.call_args[0][0]
    assert call_args["sl"] == 1.0900  # preserved
    assert call_args["tp"] == 1.1200

    print("Modify position test passed!\n")

In [7]:
test_position_manager_modify()

2026-06-21 20:00:48,953 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-21 20:00:48,959 [INFO] PositionManager: MODIFY SUCCESS: Ticket 123456 SL:1.092 TP:1.11
2026-06-21 20:00:48,961 [INFO] PositionManager: MODIFY SUCCESS: Ticket 123456 SL:1.09 TP:1.12


--- Testing PositionManager.modify_position ---
Modify position test passed!



## 5. Testing DrawdownManager

Module 5 enforces daily (3%) and total (10%) drawdown ceilings. We reuse the same mocked `mt5` object from section 1 and add mocks for `account_info`, `symbol_info_tick`, and `symbol_select`. Each test below corresponds directly to a scenario from the Module 5 design spec, asserting on dollar amounts rather than just re-running the file's own `__main__` block.

In [8]:
from drawdown import DrawdownManager

STATE_FILE = "test_drawdown_state.json"

def cleanup_state_file():
    for f in [STATE_FILE, STATE_FILE + ".tmp"]:
        if os.path.exists(f):
            os.remove(f)

def set_server_date(date_str):
    """Sets the mocked tick time (Unix timestamp) returned by symbol_info_tick.

    Builds the timestamp explicitly in UTC. drawdown.py reads the tick time via
    `datetime.fromtimestamp(tick.time, tz=timezone.utc)`, so the mock must produce
    a UTC-anchored timestamp here too. Using `time.mktime(time.strptime(...))`
    instead would interpret the date as LOCAL time before converting to a Unix
    timestamp -- on any machine east of UTC (e.g. Europe/Berlin) that silently
    shifts the date back by one day once drawdown.py reinterprets it as UTC.
    """
    dt_utc = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    mt5.symbol_info_tick.return_value = MagicMock(time=dt_utc.timestamp())

def set_balance(balance):
    mt5.account_info.return_value = MagicMock(balance=balance)

def make_tracker(open_risk=0.0, positions=None):
    tracker = MagicMock()
    tracker.get_open_risk.return_value = open_risk
    tracker.get_open_positions.return_value = positions or []
    return tracker

### 5.1 Initial State — Day 1, no positions, no losses

$10,000 initial balance, daily limit 3% ($300), total limit 10% ($1,000). With nothing open and nothing lost, `max_risk_pct` should equal the daily limit (the tighter of the two ceilings on day one).

In [9]:
def test_initial_state():
    cleanup_state_file()
    mt5.symbol_select.return_value = True
    set_server_date("2024-05-20")
    set_balance(10000.0)
    tracker = make_tracker(open_risk=0.0)

    dm = DrawdownManager(
        initial_balance=10000.0,
        position_tracker=tracker,
        daily_limit_pct=0.03,
        total_limit_pct=0.10,
        state_file=STATE_FILE,
    )

    print(f"trading_allowed={dm.trading_allowed()}, max_risk_pct={dm.max_risk_pct():.4%}")
    assert dm.trading_allowed() == True
    assert abs(dm.max_risk_pct() - 0.03) < 1e-6
    assert dm.snapshot_date == "2024-05-20"
    assert dm.start_of_day_balance == 10000.0
    print("PASS: initial state")
    return dm, tracker

dm, tracker = test_initial_state()

2026-06-21 20:00:52,589 [INFO] DrawdownManager: Day boundary detected. New snapshot: date=2024-05-20, balance=10000.0


trading_allowed=True, max_risk_pct=3.0000%
PASS: initial state


### 5.2 Open Risk Reduces Remaining Budget

Open a position with $200 risk on the same $10,000 balance. Daily remaining should drop from 3% to 1% (300 - 200 = 100, i.e. 1%). No realized loss yet, so this is purely the open-risk term.

In [10]:
def test_open_risk_reduces_budget(dm, tracker):
    tracker.get_open_risk.return_value = 200.0
    dm.check()

    print(f"trading_allowed={dm.trading_allowed()}, max_risk_pct={dm.max_risk_pct():.4%}")
    assert dm.trading_allowed() == True
    assert abs(dm.max_risk_pct() - 0.01) < 1e-6
    print("PASS: open risk reduces remaining budget")

test_open_risk_reduces_budget(dm, tracker)

trading_allowed=True, max_risk_pct=1.0000%
PASS: open risk reduces remaining budget


### 5.3 Realized Loss + Open Risk Together Breach Daily Limit

Balance drops to $9,850 (a $150 realized loss) while the $200 open-risk position is still open. This is the scenario you specified directly: a position still open at full risk, combined with a prior realized loss, must be able to push the account over the daily ceiling — trading should be blocked even though no single number alone exceeds the limit.

In [11]:
def test_realized_loss_plus_open_risk_breaches(dm, tracker):
    set_balance(9850.0)
    dm.check()

    # open_risk_pct = 200 / 9850 = 0.020305...
    # daily_loss_pct = (10000 - 9850) / 10000 = 0.015
    # total_committed_daily = 0.015 + 0.020305 = 0.035305 > 0.03 limit
    print(f"trading_allowed={dm.trading_allowed()}, max_risk_pct={dm.max_risk_pct():.4%}")
    assert dm.trading_allowed() == False
    assert dm.max_risk_pct() < 0
    print("PASS: realized loss + open risk together breach the daily limit")

test_realized_loss_plus_open_risk_breaches(dm, tracker)

2026-06-21 20:00:56,684 [WARNING] DrawdownManager: Drawdown Limit Breached! Trading blocked. Max Risk Remaining: -0.5305%


trading_allowed=False, max_risk_pct=-0.5305%
PASS: realized loss + open risk together breach the daily limit


### 5.4 Closing the Position Restores Budget (Partial Recovery)

Position closes, open risk drops to $0, balance stays at $9,850. The $150 realized loss remains, but the open-risk term is gone, so trading should resume with the remaining daily budget (3% - 1.5% = 1.5%).

In [12]:
def test_close_position_recovers_budget(dm, tracker):
    tracker.get_open_risk.return_value = 0.0
    dm.check()

    print(f"trading_allowed={dm.trading_allowed()}, max_risk_pct={dm.max_risk_pct():.4%}")
    assert dm.trading_allowed() == True
    assert abs(dm.max_risk_pct() - 0.015) < 1e-6
    print("PASS: closing the position recovers remaining budget")

test_close_position_recovers_budget(dm, tracker)

2026-06-21 20:01:00,734 [INFO] DrawdownManager: Drawdown Limit Recovered. Trading permitted. Max Risk Remaining: 1.5000%


trading_allowed=True, max_risk_pct=1.5000%
PASS: closing the position recovers remaining budget


### 5.5 Day Rollover Resets the Daily Ceiling but Keeps the Total Ceiling

Server date advances to 2024-05-21. The $9,850 balance becomes the new `start_of_day_balance`, so `daily_loss_pct` resets to 0 and the daily ceiling reopens to the full 3%. The total ceiling, however, is still measured against the original $10,000 `initial_balance`, so the prior $150 loss still counts there — `max_risk_pct` should remain capped at 3% (the daily ceiling is still the tighter one).

In [13]:
def test_day_rollover_resets_daily_not_total(dm, tracker):
    set_server_date("2024-05-21")
    dm.check()

    print(f"trading_allowed={dm.trading_allowed()}, max_risk_pct={dm.max_risk_pct():.4%}, "
          f"start_of_day_balance={dm.start_of_day_balance}")
    assert dm.start_of_day_balance == 9850.0
    assert dm.snapshot_date == "2024-05-21"
    assert abs(dm.daily_loss_pct() - 0.0) < 1e-6
    # total_loss_pct = (10000 - 9850) / 10000 = 0.015, remaining_total = 0.10 - 0.015 = 0.085
    assert abs(dm.remaining_total_risk_pct() - 0.085) < 1e-6
    assert abs(dm.max_risk_pct() - 0.03) < 1e-6
    print("PASS: day rollover resets daily ceiling, total ceiling persists")

test_day_rollover_resets_daily_not_total(dm, tracker)
cleanup_state_file()

2026-06-21 20:01:01,889 [INFO] DrawdownManager: Day boundary detected. New snapshot: date=2024-05-21, balance=9850.0


trading_allowed=True, max_risk_pct=3.0000%, start_of_day_balance=9850.0
PASS: day rollover resets daily ceiling, total ceiling persists


### 5.6 Mid-Day Restart Reloads Persisted Balance — No Reset Loophole

This is the specific requirement you set: restarting the bot must **not** reset the daily risk budget. We snapshot a start-of-day balance, simulate a realized loss, then construct a **brand new** `DrawdownManager` instance against the same `state_file` and same server date — mimicking a process restart. The new instance must reload the persisted `start_of_day_balance` rather than re-snapshotting the now-lower current balance, otherwise restarting the bot would silently wipe out the day's accumulated loss tracking.

In [14]:
def test_restart_reloads_persisted_balance_no_loophole():
    cleanup_state_file()
    mt5.symbol_select.return_value = True
    set_server_date("2024-06-01")
    set_balance(10000.0)
    tracker1 = make_tracker(open_risk=0.0)

    # First instance: snapshots $10,000 as start_of_day_balance, persists to disk.
    dm1 = DrawdownManager(
        initial_balance=10000.0,
        position_tracker=tracker1,
        daily_limit_pct=0.03,
        total_limit_pct=0.10,
        state_file=STATE_FILE,
    )
    assert dm1.start_of_day_balance == 10000.0
    assert os.path.exists(STATE_FILE), "State file should have been persisted on first snapshot"

    # Simulate a realized loss before the bot "restarts" (still same server day).
    set_balance(9700.0)

    # Second instance, same state_file, same server date — simulates a process restart.
    tracker2 = make_tracker(open_risk=0.0)
    dm2 = DrawdownManager(
        initial_balance=10000.0,
        position_tracker=tracker2,
        daily_limit_pct=0.03,
        total_limit_pct=0.10,
        state_file=STATE_FILE,
    )

    print(f"After restart: start_of_day_balance={dm2.start_of_day_balance}, "
          f"daily_loss_pct={dm2.daily_loss_pct():.4%}")

    # The loophole this guards against: a naive implementation would re-snapshot
    # the CURRENT ($9,700) balance as the new start-of-day on restart, making the
    # prior loss invisible. Correct behavior reloads the persisted $10,000.
    assert dm2.start_of_day_balance == 10000.0, "Restart must reload persisted balance, not re-snapshot current balance"
    assert dm2.snapshot_date == "2024-06-01"
    # daily_loss_pct = (10000 - 9700) / 10000 = 0.03 -> exactly at the limit
    assert abs(dm2.daily_loss_pct() - 0.03) < 1e-6
    assert dm2.trading_allowed() == False  # at/over the 3% daily ceiling
    print("PASS: mid-day restart reloads persisted balance, no reset loophole")

    cleanup_state_file()

test_restart_reloads_persisted_balance_no_loophole()

2026-06-21 20:01:04,823 [INFO] DrawdownManager: Day boundary detected. New snapshot: date=2024-06-01, balance=10000.0
2026-06-21 20:01:04,835 [INFO] DrawdownManager: Drawdown state restored: date=2024-06-01, start_balance=10000.0
2026-06-21 20:01:04,835 [WARNING] DrawdownManager: Drawdown Limit Breached! Trading blocked. Max Risk Remaining: 0.0000%


After restart: start_of_day_balance=10000.0, daily_loss_pct=3.0000%
PASS: mid-day restart reloads persisted balance, no reset loophole


### 5.7 First Run — No State File Present

On a genuinely first run (no `state_file` on disk yet), the manager should snapshot the current balance immediately rather than failing or waiting.

In [15]:
def test_first_run_no_state_file():
    cleanup_state_file()
    assert not os.path.exists(STATE_FILE)

    mt5.symbol_select.return_value = True
    set_server_date("2024-07-01")
    set_balance(5000.0)
    tracker = make_tracker(open_risk=0.0)

    dm = DrawdownManager(
        initial_balance=5000.0,
        position_tracker=tracker,
        daily_limit_pct=0.03,
        total_limit_pct=0.10,
        state_file=STATE_FILE,
    )

    assert dm.start_of_day_balance == 5000.0
    assert dm.snapshot_date == "2024-07-01"
    assert os.path.exists(STATE_FILE)
    print("PASS: first run snapshots immediately and persists state")
    cleanup_state_file()

test_first_run_no_state_file()

2026-06-21 20:01:09,897 [INFO] DrawdownManager: Day boundary detected. New snapshot: date=2024-07-01, balance=5000.0


PASS: first run snapshots immediately and persists state


### 5.8 LiteFinance Symbol Fallback (`_o` suffix)

With zero open positions, `_get_server_date()` must fall back to broker-correct symbol names. LiteFinance requires the `_o` suffix (e.g. `EURUSD_o`), not the generic `EURUSD`/`GBPUSD_i` names. This confirms the fallback list resolves via `symbol_select` even when no position-derived symbol is available.

In [16]:
def test_litefinance_symbol_fallback():
    cleanup_state_file()

    # Only the LiteFinance-suffixed symbol resolves; everything else fails selection.
    def selective_select(symbol, enable):
        return symbol == "EURUSD_o"
    mt5.symbol_select.side_effect = selective_select

    set_server_date("2024-08-01")
    set_balance(7000.0)
    tracker = make_tracker(open_risk=0.0, positions=[])  # no open positions

    dm = DrawdownManager(
        initial_balance=7000.0,
        position_tracker=tracker,
        daily_limit_pct=0.03,
        total_limit_pct=0.10,
        state_file=STATE_FILE,
    )

    assert dm.snapshot_date == "2024-08-01", "Should have resolved server date via EURUSD_o fallback"
    print("PASS: resolves server date via LiteFinance '_o' suffixed fallback symbol")

    # Reset side_effect / mock state for subsequent cells
    mt5.symbol_select.side_effect = None
    mt5.symbol_select.return_value = True
    cleanup_state_file()

test_litefinance_symbol_fallback()

2026-06-21 20:01:12,924 [INFO] DrawdownManager: Day boundary detected. New snapshot: date=2024-08-01, balance=7000.0


PASS: resolves server date via LiteFinance '_o' suffixed fallback symbol


### 5.9 No Symbol Resolves — Fails Safe (No Spurious Snapshot)

If every fallback symbol fails to resolve (e.g. transient connection issue), `_get_server_date()` returns `None`. Per the spec, this must **not** trigger a re-snapshot — failing safe means skipping the boundary check that cycle, not risking a spurious day-rollover reset.

In [17]:
def test_no_symbol_resolves_fails_safe():
    cleanup_state_file()

    # First, establish a valid snapshot normally.
    mt5.symbol_select.return_value = True
    set_server_date("2024-09-01")
    set_balance(8000.0)
    tracker = make_tracker(open_risk=0.0)

    dm = DrawdownManager(
        initial_balance=8000.0,
        position_tracker=tracker,
        daily_limit_pct=0.03,
        total_limit_pct=0.10,
        state_file=STATE_FILE,
    )
    assert dm.snapshot_date == "2024-09-01"
    original_start_balance = dm.start_of_day_balance

    # Now simulate total symbol resolution failure (e.g. connection hiccup) AND
    # a balance change that would be wrongly captured if a spurious snapshot occurred.
    mt5.symbol_select.return_value = False
    mt5.symbol_info_tick.return_value = None
    mt5.copy_rates_from_pos.return_value = None
    set_balance(7500.0)

    dm.check()

    print(f"start_of_day_balance after failed resolution: {dm.start_of_day_balance} "
          f"(should be unchanged from {original_start_balance})")
    assert dm.start_of_day_balance == original_start_balance, "Must not re-snapshot when server date cannot be resolved"
    assert dm.snapshot_date == "2024-09-01"
    print("PASS: fails safe — no spurious snapshot when no symbol resolves")

    # Restore mocks for any subsequent cells
    mt5.symbol_select.return_value = True
    mt5.symbol_info_tick.side_effect = None
    cleanup_state_file()

test_no_symbol_resolves_fails_safe()

2026-06-21 20:01:14,247 [INFO] DrawdownManager: Day boundary detected. New snapshot: date=2024-09-01, balance=8000.0
2026-06-21 20:01:14,250 [WARNING] DrawdownManager: Could not retrieve MT5 server date for boundary check.
2026-06-21 20:01:14,251 [WARNING] DrawdownManager: Drawdown Limit Breached! Trading blocked. Max Risk Remaining: -3.2500%


start_of_day_balance after failed resolution: 8000.0 (should be unchanged from 8000.0)
PASS: fails safe — no spurious snapshot when no symbol resolves


### 5.10 All DrawdownManager Tests Passed

If every cell above ran without an `AssertionError`, Module 5 is behaving per spec: persistence matches `position_tracker.py`'s atomic-write pattern, the daily/total dual-ceiling math is correct, the restart loophole is closed, and the LiteFinance symbol fallback resolves server date without live positions.

## 6. Testing PositionSizer

Module 6 computes lot size from a risk percentage and SL distance. It is a pure calculator — no MT5 connection state, no decision-making about whether to trade. We mock `mt5.symbol_info(symbol)` per-case, following the same pattern used in the module's own `__main__` block, but split combined validation branches into separate test cases and add coverage for precision drift and a realistic JPY-pair contract spec.

In [18]:
from risk_sizing import PositionSizer

sizer = PositionSizer()

def mock_symbol_info(trade_contract_size=100000, volume_step=0.01, volume_min=0.01, volume_max=100.0):
    return MagicMock(
        trade_contract_size=trade_contract_size,
        volume_step=volume_step,
        volume_min=volume_min,
        volume_max=volume_max,
    )

### 6.1 Happy Path — Exact Lot Size, No Rounding Needed

1% risk on $10,000 = $100. Entry 1.1000, SL 1.0950 (50 pip / 0.0050 distance) on a standard 100,000 contract size. `raw_lot = 100 / (0.0050 * 100000) = 0.2` — lands exactly on a `volume_step` multiple, so no rounding should change the result.

In [19]:
def test_happy_path_exact_lot():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = mock_symbol_info()
        res = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, 0.01, 10000.0)

        print(res)
        assert res["success"] == True
        assert res["lot_size"] == 0.2
        assert abs(res["risk_dollars"] - 100.0) < 1e-9
        assert abs(res["risk_pct_actual"] - 0.01) < 1e-9
        assert res["capped_at_max"] == False
        assert res["error"] is None
        print("PASS: happy path, exact lot size")

test_happy_path_exact_lot()

2026-06-21 20:01:20,989 [INFO] PositionSizer: SUCCESS: EURUSD lot_size=0.2 risk=$100.00 (1.00%)


{'success': True, 'lot_size': 0.2, 'risk_dollars': 100.00000000000232, 'risk_pct_actual': 0.010000000000000231, 'capped_at_max': False, 'error': None}
PASS: happy path, exact lot size


### 6.2 Round-Down Reduces Actual Risk Below Requested

$151 risk (1.51% of $10,000) against the same 0.0050 SL distance gives `raw_lot = 0.302`, which is not a multiple of `volume_step=0.01`. Rounding down must land on `0.30`, and the resulting `risk_pct_actual` must come out strictly **less than** the requested `0.0151` — confirming round-down never lets actual risk exceed what was asked for.

In [20]:
def test_round_down_reduces_actual_risk():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = mock_symbol_info()
        res = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, 0.0151, 10000.0)

        print(res)
        assert res["success"] == True
        assert res["lot_size"] == 0.3
        assert abs(res["risk_dollars"] - 150.0) < 1e-9
        assert res["risk_pct_actual"] < 0.0151, "Round-down must never produce MORE risk than requested"
        print("PASS: round-down reduces actual risk below requested")

test_round_down_reduces_actual_risk()

2026-06-21 20:01:25,313 [INFO] PositionSizer: SUCCESS: EURUSD lot_size=0.3 risk=$150.00 (1.50%)


{'success': True, 'lot_size': 0.3, 'risk_dollars': 150.00000000000344, 'risk_pct_actual': 0.015000000000000345, 'capped_at_max': False, 'error': None}
PASS: round-down reduces actual risk below requested


### 6.3 Floating-Point Precision Drift at a Round-Step Boundary

This targets the specific guard in the code: `round(raw_lot / volume_step, 10)` before flooring. Without that inner `round()`, a value that *should* land exactly on a step multiple (e.g. `20.0`) can instead compute as `19.999999999996` due to float division, and `math.floor()` would then silently produce one step too few. We pick risk/SL/contract-size inputs designed to land close to an exact multiple of `volume_step` and confirm the result is the full multiple, not one step short.

In [21]:
def test_precision_drift_at_step_boundary():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = mock_symbol_info(
            trade_contract_size=100000, volume_step=0.01, volume_min=0.01, volume_max=100.0
        )
        # SL distance 0.0010, risk_pct chosen so raw_lot lands exactly on 0.20
        # raw_lot = risk_dollars / (sl_distance * contract_size)
        # risk_dollars = 0.20 * 0.0010 * 100000 = $20
        # risk_pct = 20 / 10000 = 0.002 -> raw_lot should be exactly 0.20 (20 steps of 0.01)
        res = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0990, 0.002, 10000.0)

        print(res)
        assert res["success"] == True
        # If precision drift caused premature floor, this would come out as 0.19 instead of 0.20
        assert res["lot_size"] == 0.2, f"Expected 0.2 exactly, got {res['lot_size']} (possible precision drift)"
        print("PASS: lands on exact step boundary, no precision-drift underflow")

test_precision_drift_at_step_boundary()

2026-06-21 20:01:27,113 [INFO] PositionSizer: SUCCESS: EURUSD lot_size=0.2 risk=$20.00 (0.20%)


{'success': True, 'lot_size': 0.2, 'risk_dollars': 20.000000000002238, 'risk_pct_actual': 0.002000000000000224, 'capped_at_max': False, 'error': None}
PASS: lands on exact step boundary, no precision-drift underflow


### 6.4 Below Minimum Lot — Fails, Does NOT Auto-Bump

Per the locked design decision: if the calculated lot is below `volume_min`, the module must return failure rather than silently sizing up to the minimum (which would risk more than requested).

In [22]:
def test_below_minimum_fails_no_autobump():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = mock_symbol_info(volume_min=0.1)
        # $10 risk, 0.0050 SL -> raw_lot = 0.02, below volume_min=0.1
        res = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, 0.001, 10000.0)

        print(res)
        assert res["success"] == False
        assert res["error"] == "lot_size_below_minimum"
        assert res["lot_size"] == 0.0, "Must return 0.0, not auto-bump to volume_min"
        print("PASS: below-minimum lot fails cleanly, no auto-bump")

test_below_minimum_fails_no_autobump()

2026-06-21 20:01:31,400 [ERROR] PositionSizer: Lot size 0.02 below minimum 0.1 for EURUSD


{'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'lot_size_below_minimum'}
PASS: below-minimum lot fails cleanly, no auto-bump


### 6.5 Capped at Maximum — Succeeds with Reduced Risk

Unlike the minimum case, exceeding `volume_max` is **not** a failure: capping down to `volume_max` only ever reduces risk below what was requested, so it's safe. `capped_at_max` should be `True` and `success` should remain `True`.

In [23]:
def test_capped_at_maximum_succeeds():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = mock_symbol_info(volume_max=1.0)
        # 10% risk on $10,000 = $1000, 0.0050 SL -> raw_lot = 2.0, capped to volume_max=1.0
        res = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, 0.1, 10000.0)

        print(res)
        assert res["success"] == True
        assert res["capped_at_max"] == True
        assert res["lot_size"] == 1.0
        assert abs(res["risk_dollars"] - 500.0) < 1e-9
        assert res["risk_pct_actual"] < 0.1, "Capping must reduce actual risk below requested"
        print("PASS: capped at maximum, succeeds with reduced (not increased) risk")

test_capped_at_maximum_succeeds()

2026-06-21 20:01:32,400 [WARNING] PositionSizer: Lot size 2.0 capped at maximum 1.0 for EURUSD
2026-06-21 20:01:32,401 [INFO] PositionSizer: SUCCESS: EURUSD lot_size=1.0 risk=$500.00 (5.00%)


{'success': True, 'lot_size': 1.0, 'risk_dollars': 500.00000000001154, 'risk_pct_actual': 0.050000000000001155, 'capped_at_max': True, 'error': None}
PASS: capped at maximum, succeeds with reduced (not increased) risk


### 6.6 Symbol Info Unavailable

`mt5.symbol_info()` returns `None` for an unknown or unselected symbol — the module must fail cleanly rather than raise an `AttributeError` trying to read `.trade_contract_size` off `None`.

In [24]:
def test_symbol_info_unavailable():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = None
        res = sizer.calculate_lot_size("UNKNOWN_SYMBOL", 1.1000, 1.0950, 0.01, 10000.0)

        print(res)
        assert res["success"] == False
        assert res["error"] == "symbol_info_unavailable"
        assert res["lot_size"] == 0.0
        print("PASS: symbol info unavailable fails cleanly")

test_symbol_info_unavailable()

2026-06-21 20:01:34,646 [ERROR] PositionSizer: Symbol info unavailable for UNKNOWN_SYMBOL


{'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'symbol_info_unavailable'}
PASS: symbol info unavailable fails cleanly


### 6.7 Invalid SL Distance (Entry Equals SL)

Entry price equal to SL price means zero distance — dividing by this would raise `ZeroDivisionError`. This check must fire *before* any MT5 call, so we assert `mt5.symbol_info` is never even invoked (catches the latent issue where this check could silently rely on an unconfigured mock if validation order ever changed).

In [25]:
def test_invalid_sl_distance_short_circuits_before_mt5_call():
    with patch("MetaTrader5.symbol_info") as mock_info:
        res = sizer.calculate_lot_size("EURUSD", 1.1000, 1.1000, 0.01, 10000.0)

        print(res)
        assert res["success"] == False
        assert res["error"] == "invalid_sl_distance"
        assert res["lot_size"] == 0.0
        # Confirms this check truly short-circuits before any MT5 access --
        # if validation order were ever swapped, this assertion would catch it
        # even though mock_info was never given a return_value.
        mock_info.assert_not_called()
        print("PASS: invalid SL distance fails before any MT5 symbol_info call")

test_invalid_sl_distance_short_circuits_before_mt5_call()

2026-06-21 20:01:35,698 [ERROR] PositionSizer: Invalid SL distance: entry=1.1, sl=1.1


{'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'invalid_sl_distance'}
PASS: invalid SL distance fails before any MT5 symbol_info call


### 6.8 Invalid `risk_pct` and Invalid `account_balance` — Tested Separately

The module's own self-test only exercises `risk_pct <= 0 OR account_balance <= 0` as a combined branch. Here we test each input independently (one zero/negative at a time, the other valid) to confirm both halves of the condition actually work, not just the case where both happen to be invalid together.

In [26]:
def test_invalid_risk_pct_and_balance_separately():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = mock_symbol_info()

        # risk_pct invalid (zero), account_balance valid
        res_zero_risk = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, 0.0, 10000.0)
        print("zero risk_pct:", res_zero_risk)
        assert res_zero_risk["success"] == False
        assert res_zero_risk["error"] == "invalid_input"

        # risk_pct invalid (negative), account_balance valid
        res_negative_risk = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, -0.01, 10000.0)
        print("negative risk_pct:", res_negative_risk)
        assert res_negative_risk["success"] == False
        assert res_negative_risk["error"] == "invalid_input"

        # account_balance invalid (zero), risk_pct valid
        res_zero_balance = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, 0.01, 0.0)
        print("zero account_balance:", res_zero_balance)
        assert res_zero_balance["success"] == False
        assert res_zero_balance["error"] == "invalid_input"

        # account_balance invalid (negative), risk_pct valid
        res_negative_balance = sizer.calculate_lot_size("EURUSD", 1.1000, 1.0950, 0.01, -5000.0)
        print("negative account_balance:", res_negative_balance)
        assert res_negative_balance["success"] == False
        assert res_negative_balance["error"] == "invalid_input"

        print("PASS: invalid risk_pct and invalid account_balance both fail independently")

test_invalid_risk_pct_and_balance_separately()

2026-06-21 20:01:36,578 [ERROR] PositionSizer: Invalid input: risk_pct=0.0, account_balance=10000.0
2026-06-21 20:01:36,579 [ERROR] PositionSizer: Invalid input: risk_pct=-0.01, account_balance=10000.0
2026-06-21 20:01:36,580 [ERROR] PositionSizer: Invalid input: risk_pct=0.01, account_balance=0.0
2026-06-21 20:01:36,581 [ERROR] PositionSizer: Invalid input: risk_pct=0.01, account_balance=-5000.0


zero risk_pct: {'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'invalid_input'}
negative risk_pct: {'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'invalid_input'}
zero account_balance: {'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'invalid_input'}
negative account_balance: {'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'invalid_input'}
PASS: invalid risk_pct and invalid account_balance both fail independently


### 6.9 Realistic JPY-Pair Contract Specs

All prior cases use EURUSD-style 5-digit pricing. `USDJPY` (one of your actual trading symbols) has very different price scale (2-3 decimal places, ~150.00 rather than ~1.1000) and the same standard 100,000 contract size, but a much larger pip distance in raw price terms. This confirms the sizing math holds up across a structurally different symbol, not just the pair used in every prior test.

In [27]:
def test_jpy_pair_realistic_specs():
    with patch("MetaTrader5.symbol_info") as mock_info:
        mock_info.return_value = mock_symbol_info(
            trade_contract_size=100000, volume_step=0.01, volume_min=0.01, volume_max=50.0
        )
        # USDJPY: entry 150.00, SL 149.50 -> SL distance = 0.50 (50 pips at 2-decimal JPY pricing)
        # 1% risk on $10,000 = $100
        # raw_lot = 100 / (0.50 * 100000) = 100 / 50000 = 0.002 -> rounds down to 0.00 -> below volume_min
        # This SL distance is too wide for this risk budget at this balance, which is realistic
        # and should correctly fail rather than silently underflow to a wrong lot size.
        res_small = sizer.calculate_lot_size("USDJPY", 150.00, 149.50, 0.01, 10000.0)
        print("Small balance, wide SL:", res_small)
        assert res_small["success"] == False
        assert res_small["error"] == "lot_size_below_minimum"

        # Larger account balance, same SL distance -> should now succeed
        # raw_lot = (100000 * 0.01) / (0.50 * 100000) = 1000 / 50000 = 0.02
        res_larger = sizer.calculate_lot_size("USDJPY", 150.00, 149.50, 0.01, 100000.0)
        print("Larger balance, same SL:", res_larger)
        assert res_larger["success"] == True
        assert res_larger["lot_size"] == 0.02
        assert abs(res_larger["risk_dollars"] - 1000.0) < 1e-9

        print("PASS: JPY-pair specs (different price scale, same contract size) size correctly")

test_jpy_pair_realistic_specs()

2026-06-21 20:01:37,307 [ERROR] PositionSizer: Lot size 0.0 below minimum 0.01 for USDJPY
2026-06-21 20:01:37,307 [INFO] PositionSizer: SUCCESS: USDJPY lot_size=0.02 risk=$1000.00 (1.00%)


Small balance, wide SL: {'success': False, 'lot_size': 0.0, 'risk_dollars': 0.0, 'risk_pct_actual': 0.0, 'capped_at_max': False, 'error': 'lot_size_below_minimum'}
Larger balance, same SL: {'success': True, 'lot_size': 0.02, 'risk_dollars': 1000.0, 'risk_pct_actual': 0.01, 'capped_at_max': False, 'error': None}
PASS: JPY-pair specs (different price scale, same contract size) size correctly


### 6.10 All PositionSizer Tests Passed

If every cell above ran without an `AssertionError`, Module 6 is behaving per spec: round-down never exceeds requested risk, minimum-lot failures don't silently auto-bump, maximum-lot capping is treated as a safe success rather than a failure, precision drift at step boundaries doesn't cause underflow, and both halves of the `invalid_input` validation work independently — including on a structurally different symbol (JPY pricing) than the EURUSD examples in the module's own self-test.